# Task 7: ResNet-50 Style Residual Bottleneck Block and Grouped Convolutions

## Objective

Construct a custom ResNet-50-style bottleneck residual block using:

- 1×1 projection convolution
- Grouped/depthwise convolution
- 1×1 expansion convolution
- Learnable skip projection
- Batch normalization
- ReLU activation

The block is evaluated using PyTorch Profiler to compare:

- Parameter count
- Memory usage
- Floating-point operations (FLOPs)
- Execution time

Different channel scaling factors are tested to study architectural scaling.

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torch.profiler import profile, ProfilerActivity

torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Device:", device)


class BottleneckBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        base_channels,
        scale=1
    ):
        super().__init__()

        mid = int(base_channels * scale)
        out_channels = mid * 4

        # 1x1 reduction
        self.conv1 = nn.Conv2d(
            in_channels,
            mid,
            kernel_size=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(mid)

        # Depthwise convolution
        self.depthwise = nn.Conv2d(
            mid,
            mid,
            kernel_size=3,
            padding=1,
            groups=mid,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(mid)

        # 1x1 expansion
        self.conv3 = nn.Conv2d(
            mid,
            out_channels,
            kernel_size=1,
            bias=False
        )

        self.bn3 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        # Learnable skip projection
        if in_channels != out_channels:

            self.projection = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

        else:
            self.projection = nn.Identity()

    def forward(self, x):

        identity = self.projection(x)

        out = self.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        out = self.relu(
            self.bn2(
                self.depthwise(out)
            )
        )

        out = self.bn3(
            self.conv3(out)
        )

        out = out + identity

        return self.relu(out)


# Example block
block = BottleneckBlock(
    in_channels=64,
    base_channels=32
).to(device)

x = torch.randn(
    2, 64, 56, 56,
    device=device
)

y = block(x)

print("Input :", x.shape)
print("Output:", y.shape)

# Parameter Count and Channel Scaling

The same bottleneck architecture is evaluated at different channel scaling factors.

Increasing the scale increases the number of channels and therefore increases computational and memory requirements.

In [ ]:
def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
    )


scales = [0.5, 1.0, 2.0]

for scale in scales:

    model = BottleneckBlock(
        in_channels=64,
        base_channels=32,
        scale=scale
    ).to(device)

    params = count_parameters(model)

    output = model(x)

    print(
        f"Scale {scale:<3} | "
        f"Parameters: {params:,} | "
        f"Output: {tuple(output.shape)}"
    )

# PyTorch Profiler

PyTorch Profiler is used to measure execution characteristics of each channel configuration.

The profiler records CPU/CUDA activity, tensor shapes, memory usage, and FLOPs where supported by the underlying operators.

In [ ]:
def profile_model(scale):

    model = BottleneckBlock(
        64,
        32,
        scale
    ).to(device)

    model.eval()

    sample = torch.randn(
        2, 64, 56, 56,
        device=device
    )

    activities = [ProfilerActivity.CPU]

    if torch.cuda.is_available():
        activities.append(
            ProfilerActivity.CUDA
        )

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_flops=True
    ) as prof:

        with torch.no_grad():
            for _ in range(5):
                model(sample)

    params = count_parameters(model)

    total_flops = sum(
        item.flops or 0
        for item in prof.key_averages()
    )

    total_memory = max(
        [
            item.self_cpu_memory_usage
            for item in prof.key_averages()
        ],
        default=0
    )

    return (
        params,
        total_flops,
        total_memory,
        prof
    )


for scale in scales:

    params, flops, memory, prof = profile_model(
        scale
    )

    print(
        f"\nScale: {scale}"
    )
    print(
        f"Parameters : {params:,}"
    )
    print(
        f"FLOPs      : {flops:,}"
    )
    print(
        f"Peak op memory contribution: "
        f"{memory / 1024**2:.2f} MB"
    )

In [ ]:
# ============================================================
# Profiler summary for the largest configuration
# ============================================================

params, flops, memory, prof = profile_model(2.0)

print(prof.key_averages().table(
    sort_by="self_cpu_time_total",
    row_limit=10
))

print("\nLargest configuration")
print("Parameters:", f"{params:,}")
print("FLOPs:", f"{flops:,}")
print(
    "Memory:",
    f"{memory / 1024**2:.2f} MB"
)

# Conclusion

A custom ResNet-50-style bottleneck residual block was successfully implemented using PyTorch.

The architecture contains:

- 1×1 channel reduction
- Depthwise 3×3 convolution
- 1×1 channel expansion
- Batch normalization
- ReLU activation
- Learnable residual projection

The depthwise convolution reduces the number of convolutional parameters compared with a standard dense convolution.

Different channel scaling factors were profiled using PyTorch Profiler. Increasing the channel scale increases parameter count, memory requirements, and computational cost.

The experiment demonstrates how residual connections and grouped/depthwise convolutions can be combined to construct scalable and computationally efficient deep architectures.